# Clean up generated test data

⚠️ **Warning:** This permanently deletes the test data generated by this repository from the configured bucket.

Only objects under the generator folders are deleted. The cleanup leaves `personer2/` untouched. GCS folders are virtual and disappear when their objects are removed. 🗑️☁️

In [ ]:
import sys
from pathlib import Path

import gcsfs

notebooks_dir = next(
    directory
    for directory in (Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent)
    if (directory / "test_data_plan.py").exists()
)
sys.path.insert(0, str(notebooks_dir))

from test_data_plan import BUCKET, generated_top_level_prefixes

confirmation = input("Type 'delete' to permanently delete all generated test data: ")
if confirmation != "delete":
    raise RuntimeError("Cleanup cancelled. Type exactly 'delete' to continue.")

filesystem = gcsfs.GCSFileSystem()
generated_prefixes = generated_top_level_prefixes() | {"statistikk", "test-data"}
objects = [
    path
    for path in filesystem.find(BUCKET)
    if path.removeprefix(f"{BUCKET}/").split("/", 1)[0] in generated_prefixes
]
for path in objects:
    filesystem.rm(path)
    print(f"[cleanup] Removed gs://{path}")

print(f"[cleanup] Removed {len(objects)} generated files from gs://{BUCKET} ✅")